# Bayesian Statistics Practice Notebook

This notebook is a hands-on companion to the Markdown file on **Bayesian Statistics**.
It demonstrates the key concepts of Bayesian reasoning and inference.

Topics covered:

1. Bayes' theorem and a diagnostic test example
2. Beta-Binomial conjugate update
3. Prior strength and influence on the posterior
4. Normal-Normal conjugate update
5. MAP estimation vs MLE
6. Credible intervals
7. Posterior predictive distribution
8. MCMC intuition with Metropolis-Hastings
9. Bayesian regression (MAP as Ridge)
10. Bayes factors for model comparison
11. Summary table
12. Mini exercises

The notebook is designed for learning, GitHub repositories, and classroom use.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import beta as beta_dist, norm, binom
from scipy.special import betaln
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import r2_score
np.random.seed(42)

## 1. Bayes' Theorem

The central equation of Bayesian statistics:

$$P(\theta|D) = \frac{P(D|\theta)\, P(\theta)}{P(D)}$$

Components:
- **Prior** P(\theta): Belief about \theta before seeing data
- **Likelihood** P(D|\theta): How probable is the data given \theta?
- **Evidence** P(D): Normalizing constant
- **Posterior** P(\theta|D): Updated belief after seeing data

**Simplified:** Posterior \propto Likelihood x Prior

### Diagnostic Test Example

A disease has 1% prevalence. A test has 99% sensitivity and 95% specificity.
What is the probability of disease given a positive test result?

This illustrates why base rates matter in Bayesian reasoning.

In [ ]:
P_disease = 0.01
sensitivity = 0.99   # P(+ | disease)
specificity = 0.95   # P(- | no disease)

P_no_disease = 1 - P_disease
P_pos_given_disease = sensitivity
P_pos_given_no_disease = 1 - specificity

# Total probability of a positive test
P_pos = P_pos_given_disease * P_disease + P_pos_given_no_disease * P_no_disease

# Posterior via Bayes' theorem
P_disease_given_pos = (P_pos_given_disease * P_disease) / P_pos

pd.DataFrame({
    'Quantity': ['P(disease)', 'P(+|disease)', 'P(+|no disease)', 'P(+) total', 'P(disease|+)'],
    'Value': [P_disease, P_pos_given_disease, P_pos_given_no_disease, P_pos, P_disease_given_pos]
})

**Insight:** Even with a 99% sensitive test, only about 16% of positive tests indicate actual disease because the base rate is very low.
This is the base-rate neglect problem that Bayesian reasoning corrects.

## 2. Beta-Binomial Conjugate Update

For a proportion parameter p:
- **Prior:** p ~ Beta(\alpha, \beta)
- **Likelihood:** X|p ~ Binomial(n, p)
- **Posterior:** p|X ~ Beta(\alpha + x, \beta + n - x)

This is a conjugate model: prior and posterior are in the same distribution family.

Interpretation: \alpha and \beta act as pseudo-counts of successes and failures.

In [ ]:
alpha_prior, beta_prior = 2.0, 2.0
n_obs, x_successes = 30, 22

alpha_post = alpha_prior + x_successes
beta_post  = beta_prior  + n_obs - x_successes

p_grid = np.linspace(0.001, 0.999, 400)
prior_pdf    = beta_dist.pdf(p_grid, alpha_prior, beta_prior)
lik_unnorm   = binom.pmf(x_successes, n_obs, p_grid)
post_pdf     = beta_dist.pdf(p_grid, alpha_post, beta_post)

plt.figure(figsize=(9, 4))
plt.plot(p_grid, prior_pdf / prior_pdf.max(),  label='Prior (scaled)')
plt.plot(p_grid, lik_unnorm / lik_unnorm.max(), label='Likelihood (scaled)', linestyle='--')
plt.plot(p_grid, post_pdf / post_pdf.max(),    label='Posterior (scaled)', linewidth=2)
plt.title('Beta-Binomial Bayesian Update')
plt.xlabel('p')
plt.ylabel('Scaled density')
plt.legend()
plt.show()

pd.DataFrame({
    'Quantity': ['Prior mean', 'Posterior mean', 'MLE (x/n)'],
    'Value': [
        alpha_prior / (alpha_prior + beta_prior),
        alpha_post  / (alpha_post  + beta_post),
        x_successes / n_obs
    ]
})

## 3. Prior Strength and Influence

A stronger (more concentrated) prior requires more data to be overridden by the likelihood.
With the same data, we compare three priors: weak, moderate, and strong.

In [ ]:
priors_to_compare = [
    ('Weak Beta(1,1)',      1,  1),
    ('Moderate Beta(5,5)',  5,  5),
    ('Strong Beta(20,20)', 20, 20),
]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (label, a, b) in zip(axes, priors_to_compare):
    a_post = a + x_successes
    b_post = b + n_obs - x_successes
    ax.plot(p_grid, beta_dist.pdf(p_grid, a, b), label='Prior')
    ax.plot(p_grid, beta_dist.pdf(p_grid, a_post, b_post), label='Posterior')
    ax.axvline(x_successes / n_obs, linestyle=':', label='MLE')
    ax.set_title(label)
    ax.set_xlabel('p')
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

**Practice:** Which prior allows the data to dominate most strongly?

## 4. Normal-Normal Conjugate Update

For a mean parameter \mu with **known** observation variance \sigma^2:
- **Prior:** \mu ~ N(\mu_0, \tau_0^2)
- **Likelihood:** X_i|\mu ~ N(\mu, \sigma^2)
- **Posterior:** \mu|X ~ N(\mu_n, \tau_n^2)

Posterior precision adds prior and data precisions:

$$\frac{1}{\tau_n^2} = \frac{1}{\tau_0^2} + \frac{n}{\sigma^2}$$

Posterior mean is the precision-weighted average:

$$\mu_n = \tau_n^2 \left(\frac{\mu_0}{\tau_0^2} + \frac{n\bar{x}}{\sigma^2}\right)$$

In [ ]:
mu_0      = 50.0   # prior mean
tau_0_sq  = 25.0   # prior variance
sigma_sq  = 16.0   # known observation variance

data_n = np.random.normal(loc=60, scale=np.sqrt(sigma_sq), size=20)
n_n    = len(data_n)
x_bar  = data_n.mean()

tau_n_sq = 1 / (1/tau_0_sq + n_n/sigma_sq)
mu_n     = tau_n_sq * (mu_0/tau_0_sq + n_n*x_bar/sigma_sq)

theta_grid = np.linspace(30, 80, 400)
plt.figure(figsize=(9, 4))
plt.plot(theta_grid, norm.pdf(theta_grid, mu_0, np.sqrt(tau_0_sq)),  label='Prior')
plt.plot(theta_grid, norm.pdf(theta_grid, mu_n, np.sqrt(tau_n_sq)),  label='Posterior', linewidth=2)
plt.axvline(x_bar, linestyle='--', label=f'Sample mean = {x_bar:.1f}')
plt.title('Normal-Normal Bayesian Update')
plt.xlabel('mu')
plt.ylabel('Density')
plt.legend()
plt.show()

pd.DataFrame({
    'Quantity': ['Prior mean', 'Sample mean', 'Posterior mean', 'Prior std', 'Posterior std'],
    'Value':    [mu_0, x_bar, mu_n, np.sqrt(tau_0_sq), np.sqrt(tau_n_sq)]
})

## 5. MAP Estimation vs MLE

- **MLE:** Maximizes P(D|\theta) — no prior influence
- **MAP:** Maximizes P(\theta|D) \propto P(D|\theta)P(\theta) — prior regularizes

For Normal-Normal:
- MLE = sample mean \bar{x}
- MAP = posterior mean \mu_n (precision-weighted average of prior and data)

With a flat prior (\tau_0 \to \infty), MAP = MLE.
With a strong prior, MAP is pulled toward the prior mean even with moderate data.

In [ ]:
sample_sizes = [3, 5, 10, 20, 50, 100]
rows = []
for n_s in sample_sizes:
    d = np.random.normal(loc=60, scale=np.sqrt(sigma_sq), size=n_s)
    xb = d.mean()
    tau_s = 1 / (1/tau_0_sq + n_s/sigma_sq)
    mu_s  = tau_s * (mu_0/tau_0_sq + n_s*xb/sigma_sq)
    rows.append((n_s, round(xb, 2), round(mu_s, 2)))

pd.DataFrame(rows, columns=['n', 'MLE (x-bar)', 'MAP (mu_n)'])

**Observation:** With small n, MAP is pulled toward the prior mean (50). As n grows, MAP converges to MLE.

## 6. Credible Intervals

A **Bayesian credible interval** directly states:

> "With 95% probability, \theta lies in this interval."

This is computed from the posterior distribution quantiles.

Contrast with frequentist confidence intervals, which describe long-run coverage properties
and do not make probability statements about a fixed parameter.

In [ ]:
# Beta posterior from Section 2
ci_90 = beta_dist.ppf([0.05, 0.95], alpha_post, beta_post)
ci_95 = beta_dist.ppf([0.025, 0.975], alpha_post, beta_post)
ci_99 = beta_dist.ppf([0.005, 0.995], alpha_post, beta_post)
post_mean_val = alpha_post / (alpha_post + beta_post)
post_mode_val = (alpha_post - 1) / (alpha_post + beta_post - 2)

pd.DataFrame({
    'Quantity': ['Posterior mean', 'Posterior mode (MAP)', '90% CI low', '90% CI high', '95% CI low', '95% CI high', '99% CI low', '99% CI high'],
    'Value':    [post_mean_val, post_mode_val, ci_90[0], ci_90[1], ci_95[0], ci_95[1], ci_99[0], ci_99[1]]
})

## 7. Posterior Predictive Distribution

The posterior predictive answers: given what we observed, **what will new data look like?**

$$P(\tilde{x}|D) = \int P(\tilde{x}|\theta)\, P(\theta|D)\, d\theta$$

We approximate it by:
1. Sample \theta^{(i)} from the posterior
2. Sample new data \tilde{x}^{(i)} from P(\tilde{x}|\theta^{(i)})
3. The histogram of \tilde{x}^{(i)} approximates the posterior predictive

In [ ]:
n_future = 10
p_samples = beta_dist.rvs(alpha_post, beta_post, size=6000, random_state=42)
future_successes = np.random.binomial(n_future, p_samples)

plt.figure(figsize=(8, 4))
plt.hist(future_successes, bins=np.arange(-0.5, n_future + 1.5, 1), density=True)
plt.title(f'Posterior Predictive: successes in next {n_future} trials')
plt.xlabel('Number of successes')
plt.ylabel('Probability')
plt.show()

pd.DataFrame({
    'Metric': ['Predictive mean', 'Predictive std', 'P(>= 8 successes)'],
    'Value':  [future_successes.mean(), future_successes.std(ddof=1), (future_successes >= 8).mean()]
})

## 8. MCMC Intuition: Metropolis-Hastings

Markov Chain Monte Carlo (MCMC) draws samples from a posterior that is hard to compute analytically.

**Metropolis-Hastings algorithm:**
1. Start at current \theta
2. Propose \theta^* from a proposal distribution q(\theta^*|\theta)
3. Compute acceptance ratio r = P(\theta^*|D) / P(\theta|D)
4. Accept \theta^* with probability min(1, r); otherwise stay at \theta

Over many iterations, the chain visits states proportional to the posterior.

In [ ]:
def log_posterior_beta_binom(p, a_pr, b_pr, x, n):
    if p <= 0 or p >= 1:
        return -np.inf
    return (a_pr - 1)*np.log(p) + (b_pr - 1)*np.log(1-p) + x*np.log(p) + (n-x)*np.log(1-p)

n_mcmc = 6000
proposal_std = 0.06
chain = [0.5]
n_accept = 0

for _ in range(n_mcmc):
    current = chain[-1]
    proposed = current + np.random.normal(0, proposal_std)
    log_r = (log_posterior_beta_binom(proposed, alpha_prior, beta_prior, x_successes, n_obs) -
             log_posterior_beta_binom(current,  alpha_prior, beta_prior, x_successes, n_obs))
    if np.log(np.random.rand()) < log_r:
        chain.append(proposed)
        n_accept += 1
    else:
        chain.append(current)

burn_in = 1000
chain = np.array(chain[burn_in:])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(chain[:300])
axes[0].set_title('MCMC Trace (first 300 post-burn-in steps)')
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('p')

axes[1].hist(chain, bins=40, density=True, alpha=0.6, label='MCMC samples')
axes[1].plot(p_grid, beta_dist.pdf(p_grid, alpha_post, beta_post), label='Exact posterior', linewidth=2)
axes[1].set_title('MCMC vs Exact Posterior')
axes[1].set_xlabel('p')
axes[1].legend()
plt.tight_layout()
plt.show()

pd.DataFrame({
    'Quantity':  ['MCMC mean', 'Exact posterior mean', 'MCMC std', 'Exact posterior std', 'Acceptance rate'],
    'Value':     [chain.mean(), alpha_post/(alpha_post+beta_post), chain.std(ddof=1),
                  beta_dist.std(alpha_post, beta_post), n_accept/n_mcmc]
})

## 9. Bayesian Regression: MAP as Ridge

Bayesian linear regression with a zero-mean normal prior on coefficients yields the MAP estimate:

$$\theta_{MAP} = \arg\max_\theta\, [\log P(D|\theta) + \log P(\theta)]$$

With a normal prior P(\theta) = N(0, \sigma_\theta^2), the MAP solution equals **Ridge regression** with:

$$\lambda = \frac{\sigma^2}{\sigma_\theta^2}$$

This gives the Bayesian interpretation of regularization: a prior on coefficients.

In [ ]:
X_br = np.linspace(0, 10, 80).reshape(-1, 1)
y_br = 3 * X_br[:, 0] + 5 + np.random.normal(0, 2, 80)

ols   = LinearRegression().fit(X_br, y_br)
ridge = Ridge(alpha=1.0).fit(X_br, y_br)

pd.DataFrame({
    'Model':     ['OLS (MLE)', 'Ridge (MAP approx.)'],
    'Intercept': [ols.intercept_, ridge.intercept_],
    'Slope':     [ols.coef_[0],   ridge.coef_[0]],
    'R2':        [r2_score(y_br, ols.predict(X_br)), r2_score(y_br, ridge.predict(X_br))]
})

## 10. Bayes Factors for Model Comparison

A Bayes factor compares two models M_1 and M_2 by their marginal likelihoods:

$$BF_{12} = \frac{P(D|M_1)}{P(D|M_2)}$$

Interpretation (Jeffreys scale):
- BF > 10: Strong evidence for M_1
- 3 < BF < 10: Moderate evidence
- 1 < BF < 3: Weak/anecdotal evidence
- BF < 1: Evidence for M_2

For Beta-Binomial, the log marginal likelihood is:

$$\log P(D|M) = \log B(\alpha + x,\, \beta + n - x) - \log B(\alpha, \beta)$$

In [ ]:
def log_marginal_beta_binom(x, n, a, b):
    return betaln(a + x, b + n - x) - betaln(a, b)

# M1: Uniform prior Beta(1,1) — no assumption about p
# M2: Skeptical prior Beta(2,8) — assumes p is low
# M3: Informative prior Beta(5,5) — assumes p near 0.5
lml1 = log_marginal_beta_binom(x_successes, n_obs, 1, 1)
lml2 = log_marginal_beta_binom(x_successes, n_obs, 2, 8)
lml3 = log_marginal_beta_binom(x_successes, n_obs, 5, 5)

pd.DataFrame({
    'Model':           ['M1: Uniform Beta(1,1)', 'M2: Skeptical Beta(2,8)', 'M3: Centered Beta(5,5)'],
    'log P(D|M)':      [lml1, lml2, lml3],
    'BF vs M2':        [np.exp(lml1 - lml2), 1.0, np.exp(lml3 - lml2)]
})

**Interpretation:** Given x=22 out of n=30, the data strongly favors models that allow high p
(M1 and M3) over the skeptical model (M2).

## 11. Summary Table

In [ ]:
summary = pd.DataFrame({
    'Concept': [
        'P(disease | positive test)',
        'Beta posterior mean',
        'Beta posterior mode (MAP)',
        '95% credible interval',
        'Normal posterior mean',
        'MCMC posterior mean',
        'MCMC acceptance rate',
        'BF (M1 vs M2 skeptical)'
    ],
    'Value': [
        round(P_disease_given_pos, 4),
        round(post_mean_val, 4),
        round(post_mode_val, 4),
        f'[{ci_95[0]:.3f}, {ci_95[1]:.3f}]',
        round(mu_n, 3),
        round(chain.mean(), 4),
        round(n_accept / n_mcmc, 3),
        round(np.exp(lml1 - lml2), 2)
    ]
})
summary

## 12. Mini Exercises

Try these on your own:

1. Change the disease prevalence in the diagnostic test to 10% and compare the posterior probability.
2. Set the prior to Beta(50, 5) — a very strong prior — and observe how much data you need to shift the posterior.
3. Compare MAP and MLE estimates for n = 3, 10, 50 with a strong prior Beta(10, 10).
4. Compute the 80% and 99% credible intervals for the Beta posterior.
5. Run the Normal-Normal update with a prior mean of 70 (wrong direction) and see how many observations it takes to correct the posterior.
6. Modify the MCMC proposal standard deviation (try 0.01 and 0.3) and compare trace plots and acceptance rates.
7. Compute Bayes factors between a diffuse prior Beta(0.5, 0.5) and a strong prior Beta(20, 5) for different observed success rates.
8. Extend the Bayesian regression to multiple features and compare OLS, Ridge, and Lasso as MAP solutions with different priors.

These exercises are especially useful for AI, machine learning, probabilistic modeling, and decision-making under uncertainty.